This file is to experiment with setting thresholds for our data with NLP-produced similarity scores
- for now, just using some sample file(s) from get_scores()
- sectioning off data by each ..05 of similarity scores
- then, get cohen's kappas on that for comparison

In [1]:
import pandas as pd
from NLP_Eval_for_DE import scores, data
import ast
from sklearn.metrics import cohen_kappa_score, f1_score, confusion_matrix


Welcome to NLP eval for DE, version 1.0.0


In [50]:
testable_data = data.get_testable_data("Example\\inputs\\case study 1 input\\pain points full.csv")
codes = data.get_codes("Example\\inputs\\case study 1 input\\description codes.csv")
all_scores = scores.get_MPNet_scores(testable_data, codes)[1]

#split the lots of scores in the "Similarity scores" column into separate columns
all_scores_expanded = all_scores.copy()
#make this go up to 11 instead of 10
all_scores_expanded[[str(i) for i in range(1, 12)]] = pd.DataFrame(all_scores_expanded["Similarity scores"].tolist(), index=all_scores_expanded.index)
# add the "Open code(s)" column of testable_data to all_scores_expanded
all_scores_expanded["Consensus code"] = testable_data["Consensus code"].tolist()
#delete the "Similarity scores" column
all_scores_expanded = all_scores_expanded.drop(columns=["Similarity scores"])
all_scores_expanded

,Input phrase,1,2,3,4,5,6,7,8,9,10,11,Consensus code
0,cutting wood,0.275926,0.282282,0.069665,0.389838,0.148447,0.413128,0.295064,0.333296,0.359805,0.137787,0.165019,0
1,didn�t know how to use lathe,0.678873,0.391077,0.056961,0.327352,0.265672,0.615499,0.472437,0.234146,0.384051,0.192324,0.124558,1
2,Finding drill,0.325444,0.538674,0.115597,0.249212,0.257001,0.260363,0.317070,0.141597,0.616376,0.113057,0.120988,9
3,Taking out trash,0.101332,0.049162,0.114310,0.127289,0.239966,0.306315,0.176143,0.334177,0.260795,0.517487,0.202831,10
4,Finding clamp,0.273037,0.032551,0.041766,0.084932,0.445683,0.084752,0.254258,-0.015202,0.079665,0.034280,0.049979,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
394,Incorrect size gloves,0.124648,0.254763,0.472582,0.102650,0.080934,0.031354,0.097451,0.044948,0.166724,0.117139,0.102882,3
395,Uncleaned machines from previous users,0.141392,0.032854,0.094660,0.106024,0.323419,0.336343,0.183724,0.234295,0.189446,0.257238,0.192967,6
396,Machine incorrectly set up by previous user,0.253995,0.058985,0.090717,0.088641,0.273062,0.294684,0.243620,0.128408,0.218062,0.288532,0.211912,1
397,Unusable wood scarps were discarded in wrong p...,0.286140,0.114184,0.150265,0.151150,0.282056,0.351376,0.168188,0.474881,0.176893,0.317662,0.255104,8


In [ ]:
def eval_filtered(all_scores_expanded, min, max):
    # now filter based on threshold 
    # drop any rows where the highest score from the 10 scores is not between min and max
    all_scores_expanded_filtered = all_scores_expanded[
        (all_scores_expanded[[str(i) for i in range(1, 12)]].max(axis=1) >= min) 
        & (all_scores_expanded[[str(i) for i in range(1, 12)]].max(axis=1) <= max)]

    # now, get cohens kappa score for the filtered data
    ground_truths = all_scores_expanded_filtered["Consensus code"].tolist()
    predictions = all_scores_expanded_filtered[[str(i) for i in range(1, 12)]].idxmax(axis=1).tolist()
    # convert predictions from strings to ints
    predictions = [int(x) for x in predictions]
    #make these labels also go to 88 instead of 11
    f1s = f1_score(ground_truths, predictions, labels=list(range(1, 12)), average=None, zero_division=0.0) 
    kappa = cohen_kappa_score(ground_truths, predictions, weights=None, sample_weight=None)
    # get average f1 (will see later if this is ok)
    f1 = sum(f1s) / len(f1s)
    # count rows below the minimum threshold
    rows_below_min = len(all_scores_expanded[all_scores_expanded[[str(i) for i in range(1, 12)]].max(axis=1) < min])
    # also, count rows that had an "Consensus code" of 0, that also had a max score below the minimum threshold
    rows_below_min_and_zero = len(all_scores_expanded[(all_scores_expanded[[str(i) for i in range(1, 12)]].max(axis=1) < min) & (all_scores_expanded["Consensus code"] == 0)])
    return [f1, kappa, rows_below_min, rows_below_min_and_zero]

In [ ]:
rows = []
total_rows = len(all_scores_expanded)
for i, j in [(0.95, 1.0), (0.9, 0.95), (0.85, 0.9), (0.8, 0.85), (0.75, 0.8), (0.7, 0.75), (0.65, 0.7), (0.6, 0.65), (0.55, 0.6), (0.5, 0.55), (0.45, 0.5), (0.4, 0.45), (0.35, 0.4), (0.3, 0.35), (0.25, 0.3), (0.2, 0.25), (0.15, 0.2), (0.1, 0.15), (0.05, 0.1), (0.0, 0.05)]:
    results = eval_filtered(all_scores_expanded, i, j)
    rows_below_min = results[2]
    percentage_below_min = (rows_below_min / total_rows) * 100
    rows_below_min_and_zero = results[3]
    percentage_below_min_and_zero = (rows_below_min_and_zero / total_rows) * 100
    rows.append({"min": i, "max": j, "kappa": results[1], "f1": results[0], "percentage_below_min": percentage_below_min, "rows_below_min": rows_below_min, "percentage_below_min_and_zero": percentage_below_min_and_zero, "rows_below_min_and_zero": rows_below_min_and_zero})
thresholded_kappas = pd.DataFrame(rows)
thresholded_kappas.to_csv("thresholding_results\\thrKappas_MPNet_painpoints_desc.csv", index=False)
thresholded_kappas

C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:758: RuntimeW

,min,max,kappa,f1,percentage_below_min,rows_below_min,percentage_below_min_and_zero,rows_below_min_and_zero
0,0.95,1.00,NaN,0.000000,100.000000,906,73.399558,665
1,0.90,0.95,NaN,0.000000,100.000000,906,73.399558,665
2,0.85,0.90,NaN,0.000000,100.000000,906,73.399558,665
3,0.80,0.85,NaN,0.000000,100.000000,906,73.399558,665
4,0.75,0.80,NaN,0.000000,100.000000,906,73.399558,665
5,0.70,0.75,0.000000,0.000000,99.779249,904,73.178808,663
6,0.65,0.70,NaN,0.000000,99.779249,904,73.178808,663
7,0.60,0.65,0.032258,0.050000,99.116998,898,72.847682,660
8,0.55,0.60,0.000000,0.040000,98.675497,894,72.847682,660
9,0.50,0.55,0.411765,0.146667,98.123620,889,72.737307,659
